In [1]:
import statistics
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

def prepare_data() -> TensorDataset:
    X = torch.randn(10000, 128)
    y = torch.randint(0, 2, (10000,))
    dataset = TensorDataset(X, y)
    return dataset

def train():
    #pin_memory для более быстрой и асинхронной передачи данных с CPU на GPU
    dataloader = DataLoader(prepare_data(), batch_size=256, shuffle=True, pin_memory=True)

    model = nn.Sequential(
        nn.Linear(128, 512), nn.ReLU(),
        nn.Linear(512, 128), nn.ReLU(),
        nn.Linear(128, 2)
    ).cuda().train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    losses_history = []
    forward_events = []
    backward_events = []

    for batch_idx, (data, target) in enumerate(dataloader):
        # Генерируем шум сразу на GPU, чтобы избежать лишнего копирования
        noise = torch.randn(data.shape, device='cuda')

        # non_blocking=True для асинхронного копирования
        data = data.to('cuda', non_blocking=True) + noise
        target = target.to('cuda', non_blocking=True)

        optimizer.zero_grad()

        # cuda.Event для честного замера времени на GPU
        start_fw = torch.cuda.Event(enable_timing=True)
        end_fw = torch.cuda.Event(enable_timing=True)

        start_fw.record()
        output = model(data)
        loss = criterion(output, target)
        end_fw.record()
        forward_events.append((start_fw, end_fw))

        start_bw = torch.cuda.Event(enable_timing=True)
        end_bw = torch.cuda.Event(enable_timing=True)

        start_bw.record()
        loss.backward()
        end_bw.record()
        backward_events.append((start_bw, end_bw))

        optimizer.step()

        # Сохраняем только скалярное значение, отвязывая его от вычислительного графа
        losses_history.append(loss.item())

        print(f"Batch {batch_idx} loss: {loss.item():.4f}")


    torch.cuda.synchronize()

    forward_times = [s.elapsed_time(e) / 1000.0 for s, e in forward_events]
    backward_times = [s.elapsed_time(e) / 1000.0 for s, e in backward_events]

    print(f"Epoch finished, avg forward time is {statistics.mean(forward_times):.5f} s, "
          f"avg backward time is {statistics.mean(backward_times):.5f} s")

if __name__ == '__main__':
    train()

Batch 0 loss: 0.7027
Batch 1 loss: 0.7000
Batch 2 loss: 0.7002
Batch 3 loss: 0.6952
Batch 4 loss: 0.7131
Batch 5 loss: 0.7078
Batch 6 loss: 0.7041
Batch 7 loss: 0.7011
Batch 8 loss: 0.7057
Batch 9 loss: 0.6917
Batch 10 loss: 0.7072
Batch 11 loss: 0.6928
Batch 12 loss: 0.7033
Batch 13 loss: 0.6871
Batch 14 loss: 0.7096
Batch 15 loss: 0.7066
Batch 16 loss: 0.6952
Batch 17 loss: 0.6876
Batch 18 loss: 0.6899
Batch 19 loss: 0.7034
Batch 20 loss: 0.6971
Batch 21 loss: 0.6881
Batch 22 loss: 0.7012
Batch 23 loss: 0.6946
Batch 24 loss: 0.6885
Batch 25 loss: 0.6934
Batch 26 loss: 0.6961
Batch 27 loss: 0.6961
Batch 28 loss: 0.7043
Batch 29 loss: 0.6906
Batch 30 loss: 0.6934
Batch 31 loss: 0.6937
Batch 32 loss: 0.6995
Batch 33 loss: 0.6976
Batch 34 loss: 0.6915
Batch 35 loss: 0.6847
Batch 36 loss: 0.6952
Batch 37 loss: 0.6976
Batch 38 loss: 0.6901
Batch 39 loss: 0.6923
Epoch finished, avg forward time is 0.00333 s, avg backward time is 0.00242 s
